# Notebook for computing hashes, buckets and similarity values for the disk scheme using a hybrid approach

Utilizes the disk scheme

Incorporates:
* Hashing of trajectories using disk scheme
* Bucketing of hashes made from disk scheme
* Similarity computation between trajectories within buckets.
    * Both for DTW and Frechet
* Analysis of the produced bucket system

Produces:
* JSON file containing buckets
* Similarity values for trajectories within buckets


## Hybrid approach

In [1]:
import os
import sys

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from computation.similarity import *
from utils.helpers.bucket_evaluation import *
import json
import pandas as pd


Project root found: c:\Users\eivin\dev\JoonEndreLSH\masteroppgave


In [14]:
CITY = "rome" # "rome" or "porto"
MEASURE = "frechet" # "dtw" or "frechet"
BUCKETING_METHOD = "loose" # Bucketing method to use
TRUE_TRAJECTORIES = False # Use true trajectories or not

# Define parameter ranges, first value in list is parameters used for bucketing, second is parameters used for comparing trajectories with each other (using a hashed version of the trajectory) 
DIAMETER_VALUES = [1.6, 1.5]  # Example diameters
LAYERS_VALUES = [4, 5]  # Example layers
DISKS_VALUES = [50, 40]  # Example disk counts
SIZE = 300  # Example dataset sizes

In [24]:
file_path = f"../../../results_true/similarity_values/{CITY}/{MEASURE}/{CITY}-{MEASURE}-{SIZE}.csv"

# Read CSV, telling pandas to take the first column as the row labels:
true_sim_matrix_df = pd.read_csv(file_path, index_col=0)

# Function to convert values to float if possible
def convert_to_float(value):
    try:
        return float(value)
    except ValueError:
        return value

# Apply the function to each cell in the DataFrame
true_sim_matrix_df = true_sim_matrix_df.map(convert_to_float)
true_sim_matrix_df = (true_sim_matrix_df + true_sim_matrix_df.T)

true_sim_matrix_df.head(10)

,R_AAA,R_AAF,R_AAK,R_ABA,R_ABI,R_ABR,R_ABU,R_ABV,R_ACC,R_ACE,...,R_EIC,R_EIV,R_EJA,R_EJM,R_EKB,R_ELF,R_EMF,R_END,R_ENE,R_ENH
R_AAA,0.000000,0.045010,0.039027,0.017468,0.042974,0.045988,0.017693,0.024445,0.013421,0.025833,...,0.009289,0.035774,0.024664,0.024916,0.064672,0.029639,0.040734,0.037907,0.050638,0.030074
R_AAF,0.045010,0.000000,0.047677,0.045337,0.032228,0.021503,0.037043,0.032529,0.043869,0.043756,...,0.039415,0.038063,0.020531,0.053462,0.038226,0.038116,0.025689,0.024224,0.031204,0.064020
R_AAK,0.039027,0.047677,0.000000,0.030305,0.024059,0.056075,0.030043,0.023610,0.035125,0.016191,...,0.031745,0.022870,0.033869,0.018671,0.033632,0.017978,0.046275,0.029721,0.018392,0.027476
R_ABA,0.017468,0.045337,0.030305,0.000000,0.026421,0.047949,0.009329,0.016615,0.005732,0.024351,...,0.009805,0.037965,0.025294,0.021811,0.052807,0.016675,0.039354,0.031640,0.041466,0.031822
R_ABI,0.042974,0.032228,0.024059,0.026421,0.000000,0.041360,0.026132,0.021286,0.032012,0.033277,...,0.034423,0.044971,0.039662,0.031107,0.032739,0.016112,0.034725,0.016895,0.029089,0.036033
R_ABR,0.045988,0.021503,0.056075,0.047949,0.041360,0.000000,0.043356,0.043318,0.048747,0.049132,...,0.047059,0.045790,0.026564,0.066500,0.033090,0.047201,0.044475,0.038390,0.043257,0.064726
R_ABU,0.017693,0.037043,0.030043,0.009329,0.026132,0.043356,0.000000,0.013392,0.007319,0.024214,...,0.009962,0.037851,0.019329,0.023186,0.052507,0.016383,0.030051,0.025157,0.041192,0.040806
R_ABV,0.024445,0.032529,0.023610,0.016615,0.021286,0.043318,0.013392,0.000000,0.018442,0.025546,...,0.016190,0.027613,0.018620,0.024564,0.040626,0.006323,0.022782,0.018709,0.028078,0.048294
R_ACC,0.013421,0.043869,0.035125,0.005732,0.032012,0.048747,0.007319,0.018442,0.000000,0.027046,...,0.008040,0.040052,0.025183,0.024738,0.058379,0.022106,0.036761,0.030898,0.046512,0.034952
R_ACE,0.025833,0.043756,0.016191,0.024351,0.033277,0.049132,0.024214,0.025546,0.027046,0.000000,...,0.020742,0.013856,0.029670,0.028398,0.042292,0.021329,0.047905,0.030156,0.026694,0.022809


In [16]:
hashed_similarities, bucket_system = generate_disk_hash_similarity_with_bucketing_hybrid(
    city=CITY, diameter=DIAMETER_VALUES, layers=LAYERS_VALUES, disks=DISKS_VALUES, measure=MEASURE, size=SIZE, bucketing_method=BUCKETING_METHOD
    )

[[array([41.91969637, 12.48521092]), array([41.91671101, 12.49260238]), array([41.91096634, 12.49735358]), array([41.9137516 , 12.49865095]), array([41.91010667, 12.49691149]), array([41.89834215, 12.49230727]), array([41.91010667, 12.49691149]), array([41.89924252, 12.49780149]), array([41.89518482, 12.48988708]), array([41.89120622, 12.48393028]), array([41.89521901, 12.47973137]), array([41.89172001, 12.47434491])], [array([41.91430843, 12.48652641]), array([41.91477862, 12.50120212]), array([41.91430843, 12.48652641]), array([41.90329633, 12.4886847 ]), array([41.88984835, 12.48795721]), array([41.89310854, 12.48018691]), array([41.90329633, 12.4886847 ])], [array([41.9125189 , 12.48785534]), array([41.91591043, 12.48893324]), array([41.9125189 , 12.48785534]), array([41.90716561, 12.48766697]), array([41.9044    , 12.48927661]), array([41.90778547, 12.48126674]), array([41.90153099, 12.49214649]), array([41.89948509, 12.49790822]), array([41.89992876, 12.49892525]), array([41.8931

In [6]:

# hashed_similarities.head(40)
#bucket system to json
with open(f"bucket_system.json", "w") as f:
    
    json.dump(bucket_system, f)

In [17]:
hashed_similarities.head(20)

,R_AAA,R_AAF,R_AAK,R_ABA,R_ABI,R_ABR,R_ABU,R_ABV,R_ACC,R_ACE,...,R_EIC,R_EIV,R_EJA,R_EJM,R_EKB,R_ELF,R_EMF,R_END,R_ENE,R_ENH
R_AAA,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R_AAF,0.217256,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R_AAK,0.180155,0.242966,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R_ABA,0.097198,0.203761,0.132279,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R_ABI,0.000000,0.170827,0.157067,0.146751,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R_ABR,0.000000,0.000000,0.000000,0.000000,0.166546,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R_ABU,0.114983,0.195114,0.131630,0.049715,0.142506,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R_ABV,0.152311,0.173547,0.120879,0.104033,0.133041,0.210565,0.088946,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R_ACC,0.094365,0.212703,0.177104,0.069565,0.177906,0.000000,0.084699,0.135949,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R_ACE,0.166424,0.192186,0.086740,0.120189,0.154011,0.208623,0.123959,0.091145,0.162208,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
index1 = "R_AAK"
index2 = "R_AAA"
value = true_sim_matrix_df.loc[index1, index2]
print(f"The value at index ({index1}, {index2}) is: {value}")

The value at index (R_AAK, R_AAA) is: 0.0390269244873836


In [27]:
total_buckets = len(bucket_system)
buckets_with_multiple = sum(1 for trajectories in bucket_system.values() if len(trajectories) > 1)
buckets_with_single = total_buckets - buckets_with_multiple
largest_bucket_size = max(len(trajectories) for trajectories in bucket_system.values())
largest_bucket = max(bucket_system, key=lambda key: len(bucket_system[key]))
smallest_bucket = min(bucket_system, key=lambda key: len(bucket_system[key]))
smallest_bucket_size = len(bucket_system[smallest_bucket])


print(f"Total Buckets: {total_buckets}")
print(f"Largest Bucket(id): {largest_bucket}")
print(f"Buckets with more than one trajectory: {buckets_with_multiple}")
print(f"Buckets with only one trajectory: {buckets_with_single}")
print(f"Largest Bucket Size: {largest_bucket_size}")
print("smallest bucket: ", smallest_bucket_size)

# Optional: Display distribution percentages
multiple_bucket_percentage = (buckets_with_multiple / total_buckets) * 100 if total_buckets > 0 else 0
single_bucket_percentage = (buckets_with_single / total_buckets) * 100 if total_buckets > 0 else 0

print(f"Percentage of buckets with more than one trajectory: {multiple_bucket_percentage:.2f}%")
print(f"Percentage of buckets with only one trajectory: {single_bucket_percentage:.2f}%")

Total Buckets: 194
Largest Bucket(id): 140644554285789919830901673086818304522
Buckets with more than one trajectory: 193
Buckets with only one trajectory: 1
Largest Bucket Size: 167
smallest bucket:  1
Percentage of buckets with more than one trajectory: 99.48%
Percentage of buckets with only one trajectory: 0.52%


In [11]:
# THRESHOLD = 5
import numpy as np # type: ignore
THRESHOLDS = np.arange(1, 6.0, 1)  # Generates [0.5, 1.0, 1.5, ..., 5.5]

results = {
    "Precision": [],
    "Recall": [],
    "F1 Score": []
}



for treshold in THRESHOLDS:

    #Variables
    all_trajectory_names = list(hashed_similarities.keys()) # All trajectory names
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    precision = 0 
    recall = 0
    f1_Score = 0

    # Loop through all trajectory names
    for trajectory in all_trajectory_names:
        
        
        # Pred and ground truth
        predicted_similar = find_predicted_similar_trajectories(trajectory, bucket_system)
        ground_truth = get_nearest_neighbour_under_threshold(trajectory, treshold, true_sim_matrix_df).index.to_list()
        true_positives += calculate_true_positives(predicted_similar, ground_truth)
        false_positives += calculate_false_positives(predicted_similar, ground_truth)
        false_negatives += calculate_false_negatives(predicted_similar, ground_truth)
        # print("num predicted similar" , len(predicted_similar))
        # print("num ground truth" , len(ground_truth))

        # print(trajectory, "predicted similar: ", predicted_similar)
        # print(trajectory, "ground truth: ", ground_truth)
        # print("True positives: ", true_positives)
        # print("False positives: ", false_positives)
        # break
        

    # Calculate precision and recall
    precision = compute_bucket_system_precision(true_positives, false_positives)
    recall = compute_bucket_system_recall(true_positives, false_negatives)
    f1_score = compute_bucket_system_f1_score(precision, recall)

    results["Precision"].append(precision)
    results["Recall"].append(recall)
    results["F1 Score"].append(f1_score)
    
    
print(f"Bucket system statistics for city: {CITY}, measure: {MEASURE}, diameter: {DIAMETER_VALUES}, layers: {LAYERS_VALUES}, disks: {DISKS_VALUES}, size: {SIZE}")
# Create DataFrame with thresholds as columns and metrics as row indexes
df = pd.DataFrame(results, index=[f"Threshold = {t}" for t in THRESHOLDS]).T

df

Bucket system statistics for city: rome, measure: dtw, diameter: [1.6, 0.5], layers: [4, 6], disks: [50, 50], size: 100


,Threshold = 1.0,Threshold = 2.0,Threshold = 3.0,Threshold = 4.0,Threshold = 5.0
Precision,0.188480,0.600245,0.816422,0.916422,0.959559
Recall,1.000000,0.961146,0.914859,0.879558,0.857424
F1 Score,0.317179,0.738986,0.862842,0.897611,0.905621
